# 02c — Telematics Pricer (owns its calculation)
This notebook IS the telematics method: the CALC cell below computes the premium and registers it. `pricing.py` only connects (`quote`/`api_quote`). Hub `02_pricing.ipynb` and `03_main.ipynb` replay this exact cell via the loader — one source of truth.

In [ ]:
import copy, os, sys
ROOT = os.path.abspath('')
if ROOT not in sys.path: sys.path.insert(0, ROOT)

import pandas as pd
from voltvision import CFG, SCEN, N_YEARS, RateCard, describe_pricer, api_quote, gen, simulate
from voltvision import io
REGIME = 'telem'


In [ ]:
# CALC — owned by this notebook. Edit the math here; the connector only calls it.
import numpy as np
from voltvision.pricing import (register_pricer, FEATS_G, encode_features,
                                   fit_frequency, severity_table, training_book)

TELEM_INFO = {
    'label': 'GLM+Telematics',
    'params': ['expense_loading', 'risk_step', 'glm_alpha', 'train_frac', 'train_seed', 'train_book_seed'],
    'formula': 'GLM features + telematics_score, same severity/loading structure as glm',
}
TELEM_FEATS = FEATS_G + ['telematics_score']


def price_telem(book, card):
    o = book.copy()
    # Training source: SEPARATE historical book by default (seed 42,
    # window 2021-2025) so the priced book's own claims never train its
    # prices; card.train_book_seed = None restores legacy in-sample fit.
    src = training_book(card) if card.train_book_seed is not None else o
    tr = src[src['COHORT_YEAR'] == src['SIM_YEAR']].sample(frac=card.train_frac, random_state=card.train_seed)
    mdl = fit_frequency(tr, TELEM_FEATS, card.glm_alpha)
    sev, covsev = severity_table(src)
    keys = list(zip(o['COVERAGE_TYPE'].values, o['VEHICLE_TYPE'].values))
    s = np.array([sev.get(k, np.nan) if not np.isnan(sev.get(k, np.nan)) else covsev[k[0]]
                  for k in keys], float)
    prem = mdl.predict(encode_features(o, TELEM_FEATS)) * s * card.expense_loading
    prem *= card.risk_step ** o['FLOOD_RISK'].values.astype(int) * card.risk_step ** o['THEFT_RISK'].values.astype(int)
    return o.assign(FINAL_PREMIUM_SST=prem.round(2))


register_pricer('telem', price_telem, TELEM_INFO)
print('telem calc registered')


## Rule sheet (`describe_pricer('telem')`)
| Piece | Rule |
|---|---|
| Frequency | GLM features + `telematics_score` (0–100, higher = safer); same Poisson setup, same train split |
| Severity / loading | identical to GLM leg (severity by coverage×vehicle, `expense_loading`, `risk_step^flags`) |
| Reads | `telematics_score` is simulated per policy (rank-mapped to BEHAVIOR_RISK); no raw trip data needed |
| Live params | `expense_loading`, `risk_step`, `glm_alpha`, `train_frac`, `train_seed`, `train_book_seed` (default `42` → trains on a separate historical book, 2021–2025; `None` = legacy in-sample, parity checks only) — edit the card cell |


In [ ]:
print(describe_pricer(REGIME))
card = RateCard.from_cfg(CFG)
# --- tweak here, e.g.: ---
# card.expense_loading = 1.8
display(pd.DataFrame(vars(card).items(), columns=['field', 'value']))


## Request (simulated book in)

In [ ]:
SC = "MIX"
try:
    book = io.load_sim(SC)
    print(f"loaded shared sim_{SC}: {len(book)} rows")
except FileNotFoundError:
    print('shared book missing — quick inline sim')
    c = copy.deepcopy(CFG)
    c['n'] = 2000
    book = simulate(gen(c, SCEN[SC], c['seed']), c, SCEN[SC],
                    seed=c['seed'], n_years=N_YEARS, verbose=False)


## Response (API format: regime + label + card + metrics + priced book)

In [ ]:
resp = api_quote(book, REGIME, card)
print('regime:', resp['regime'], '|', resp['label'])
display(pd.DataFrame([resp['metrics']]))
display(resp['book'][['POLID', 'COVERAGE_TYPE', 'VEHICLE_TYPE', 'CLAIM_COUNT',
    'CLAIM_AMOUNT', 'FINAL_PREMIUM_SST']].head())
p = io.save_priced(SC, resp['regime'], resp['book'])
print('saved ->', p)


## Notes
- Lift over GLM comes only through the score → frequency link; severity/loading shared.
- Siblings: `02a_tariff.ipynb`, `02b_glm.ipynb` (same request/response shape).